# YOLOv8 Rose Disease Classification

This notebook demonstrates how to train a YOLOv8-cls model to classify rose leaf diseases and then perform predictions on new images.

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os
import shutil
%matplotlib inline

# Ensure project root is correct
DATA_ROOT = os.getcwd()
print(f"Project root (DATA_ROOT): {DATA_ROOT}")

## 1. Train the YOLOv8 Classification Model

We will load the pretrained `yolov8n-cls.pt` model and train it using the classification dataset structure in the `train/` and `val/` subdirectories.

In [ ]:
# Load pretrained model
model = YOLO('yolov8n-cls.pt')

# Train for 3 epochs, image size 224
results = model.train(
    data=DATA_ROOT, 
    epochs=3, 
    imgsz=224, 
    project='rose_classification', 
    name='train_session_final'
)

# After training, copy the best model into the root directory as 'best.pt'
best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
if os.path.exists(best_model_path):
    shutil.copy(best_model_path, os.path.join(DATA_ROOT, 'best.pt'))
    print(f"\nTraining complete! Best model saved to {os.path.join(DATA_ROOT, 'best.pt')}")
else:
    print("\nCould not find the best model weights. Check the 'runs' folder.")

## 2. Load the Best Model and Run Prediction
Now we'll load the trained `best.pt` model and test it on an image from the `test/` directory.

In [ ]:
# Load the manually saved best model
model_path = 'best.pt'

if os.path.exists(model_path):
    trained_model = YOLO(model_path)
    print(f"Loaded trained model: {model_path}")
    
    # Select an example image to predict
    img_path = 'test/Rose_Rust/brightened_IMG_20230703_213438.jpg'
    
    if os.path.exists(img_path):
        # Run inference
        results = trained_model(img_path)

        # Get the class names from the model and the top prediction
        names = trained_model.names
        output = results[0]
        top1_idx = output.probs.top1
        confidence = float(output.probs.top1conf)
        predicted_class = names[top1_idx]

        # Visualize the result
        im_bgr = cv2.imread(img_path)
        im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(8, 8))
        plt.imshow(im_rgb)
        plt.axis('off')
        plt.title(f"Prediction: {predicted_class} ({confidence:.2f})")
        plt.show()
    else:
        print(f"Image not found at {img_path}. Please check your path.")
else:
    print("Model not found. Run the training cell first.")

## 3. Batch Evaluation on Test Dir
Run validation on the test folder to see final metrics.

In [ ]:
if os.path.exists('best.pt'):
    val_results = trained_model.val(data=DATA_ROOT, split='test')
    print("Test set metrics:", val_results.results_dict)